# CNN-Based Image Colorization (Google Colab & VS Code Walkthrough)
### End-to-End Pipeline: From Raw CelebA to Grayscale Conversion, Preprocessing, U-Net Training & Comparison

This notebook contains the complete step-by-step workflow for the Deep Learning Grayscale-to-Color reconstruction project.
Each cell corresponds directly to a project stage suitable for live presentations and screen recordings.

## ──────────────────────────────────────────
## [STAGE 1] System Setup & GPU Verification
## ──────────────────────────────────────────

In [ ]:
# Check CUDA / GPU Availability and Install Necessary Libraries
import os
import sys
import torch
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt

# Ensure project root is in python path
sys.path.append(os.path.abspath('..'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[STAGE 1 COMPLETE] Active Computing Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

## ──────────────────────────────────────────
## [STAGE 2] Ingest Raw CelebA Dataset
## ──────────────────────────────────────────
CelebA contains over 200,000 celebrity faces. Here we ingest or generate our working set and resize to 128x128.

In [ ]:
from preprocessing.prepare_dataset import generate_sample_dataset, preprocess_directory

raw_dir = '../data/raw'
processed_dir = '../data/processed'

if not os.path.exists(raw_dir) or len(os.listdir(raw_dir)) == 0:
    print("Generating initial raw sample dataset...")
    generate_sample_dataset(raw_dir, num_samples=60, image_size=128)

preprocess_directory(raw_dir, processed_dir, image_size=128)
print(f"[STAGE 2 COMPLETE] Processed color images ready in: {processed_dir}")

## ──────────────────────────────────────────
## [STAGE 3] Convert Colored CelebA into Grayscale (B&W) Dataset
## ──────────────────────────────────────────
**Core Transformation Formula:**
$$Y = 0.299 \cdot R + 0.587 \cdot G + 0.114 \cdot B$$
This generates our core input grayscale dataset inside `data/grayscale/`.

In [ ]:
from preprocessing.create_grayscale_dataset import create_grayscale_dataset, generate_side_by_side_samples

gray_dir = '../data/grayscale'
create_grayscale_dataset(input_dir=processed_dir, output_dir=gray_dir, image_size=128)

# Display side-by-side preview
preview_file = '../outputs/grayscale_samples/rgb_vs_grayscale.png'
generate_side_by_side_samples(processed_dir, gray_dir, preview_file, num_samples=3)

preview_img = Image.open(preview_file)
plt.figure(figsize=(8, 8))
plt.imshow(preview_img)
plt.axis('off')
plt.title("Original Color vs Grayscale Dataset Preview", fontsize=14)
plt.show()
print("[STAGE 3 COMPLETE] Grayscale conversion successfully demonstrated!")

## ──────────────────────────────────────────
## [STAGE 4] Preprocessing: CIELAB Color Space & Normalization
## ──────────────────────────────────────────
We separate Lightness ($L \in [0, 1]$) from Chrominance ($a, b \in [-1, 1]$):
- $L_{norm} = L / 100.0$
- $a_{norm} = a / 128.0$
- $b_{norm} = b / 128.0$

In [ ]:
from preprocessing.color_space import rgb_to_lab, lab_to_rgb

sample_path = os.path.join(processed_dir, os.listdir(processed_dir)[0])
sample_rgb = np.array(Image.open(sample_path).convert('RGB'))

L, ab = rgb_to_lab(sample_rgb)
recon_rgb = lab_to_rgb(L, ab)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(sample_rgb); axes[0].set_title("Original RGB"); axes[0].axis('off')
axes[1].imshow(L, cmap='gray'); axes[1].set_title("L (Lightness: Input)"); axes[1].axis('off')
axes[2].imshow(ab[:, :, 0], cmap='RdYlGn_r'); axes[2].set_title("a Channel (Target 1)"); axes[2].axis('off')
axes[3].imshow(ab[:, :, 1], cmap='YlGnBu'); axes[3].set_title("b Channel (Target 2)"); axes[3].axis('off')
plt.tight_layout()
plt.show()

print(f"[STAGE 4 COMPLETE] L shape: {L.shape} | ab shape: {ab.shape}")

## ──────────────────────────────────────────
## [STAGE 5] Dataset Partitioning: 80% Train | 10% Val | 10% Test
## ──────────────────────────────────────────

In [ ]:
from data.celebA_dataset import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=processed_dir,
    batch_size=8,
    image_size=128,
    train_split=0.8,
    val_split=0.1,
    test_split=0.1,
    seed=42
)

print(f"Training Batches:   {len(train_loader)} ({len(train_loader.dataset)} samples)")
print(f"Validation Batches: {len(val_loader)} ({len(val_loader.dataset)} samples)")
print(f"Testing Batches:    {len(test_loader)} ({len(test_loader.dataset)} samples)")
print("[STAGE 5 COMPLETE] DataLoaders initialized successfully.")

## ──────────────────────────────────────────
## [STAGE 6] Model Definitions & Architectural Parameters
## ──────────────────────────────────────────
### Models:
1. **Basic CNN (Baseline):** Sequential encoder-decoder without skip connections.
2. **U-Net CNN (Primary):** 4-level encoder-decoder with 4 skip connections + Tanh activation.

In [ ]:
from models.basic_cnn import BasicCNN
from models.unet import UNet

basic_model = BasicCNN(in_channels=1, out_channels=2).to(device)
unet_model = UNet(in_channels=1, out_channels=2).to(device)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"Basic CNN Parameters: {count_params(basic_model):,}")
print(f"U-Net CNN Parameters: {count_params(unet_model):,}")
print("[STAGE 6 COMPLETE] Both models instantiated and mapped to device.")

## ──────────────────────────────────────────
## [STAGE 7] Model Training Loop & Optimization
## ──────────────────────────────────────────
- **Loss Function:** L1 Loss (Mean Absolute Error) $\to$ Prevents desaturated gray outputs
- **Optimizer:** Adam (lr=0.001, betas=(0.9, 0.999), weight_decay=1e-5)
- **Scheduler:** ReduceLROnPlateau (factor=0.5, patience=3)

In [ ]:
import torch.optim as optim
from training.loss import ColorizationLoss
from training.trainer import Trainer

criterion = ColorizationLoss(loss_type="L1")
optimizer = optim.Adam(unet_model.parameters(), lr=0.001, weight_decay=1e-5)

trainer = Trainer(
    model=unet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    model_name="unet",
    checkpoint_dir="../checkpoints",
    graphs_dir="../outputs/training_graphs",
    early_stopping_patience=5
)

# Train for 5 epochs for demonstration
history = trainer.fit(epochs=5)
print("[STAGE 7 COMPLETE] Training completed! Best checkpoint and curves saved.")

## ──────────────────────────────────────────
## [STAGE 8] Deep Learning Evaluation Metrics (MAE, PSNR, SSIM)
## ──────────────────────────────────────────

In [ ]:
from testing.evaluate import evaluate_model
from utils.checkpoint import load_checkpoint

load_checkpoint('../checkpoints/best_basic_cnn.pth', basic_model, device=device)
load_checkpoint('../checkpoints/best_unet.pth', unet_model, device=device)

print("Computing Evaluation Metrics on Unseen Test Dataset...")
basic_res = evaluate_model(basic_model, test_loader, device=device)
unet_res = evaluate_model(unet_model, test_loader, device=device)

print("\n" + "="*55)
print(f"{'Metric':<15} | {'Basic CNN':<15} | {'U-Net':<15}")
print("-"*55)
print(f"{'MAE (Lower)':<15} | {basic_res['MAE']:<15.4f} | {unet_res['MAE']:<15.4f}")
print(f"{'PSNR (Higher)':<15} | {basic_res['PSNR']:<12.2f} dB | {unet_res['PSNR']:<12.2f} dB")
print(f"{'SSIM (Higher)':<15} | {basic_res['SSIM']:<15.4f} | {unet_res['SSIM']:<15.4f}")
print("="*55)
print("[STAGE 8 COMPLETE] Quantitative evaluation finished!")

## ──────────────────────────────────────────
## [STAGE 9] Final 4-Panel Visual Comparison & Similarity Verification
## ──────────────────────────────────────────
Side-by-side comparison of:
1. **Original RGB Image** (Ground truth)
2. **Grayscale Input Image** (Lightness L)
3. **Basic CNN Prediction** (Baseline)
4. **U-Net Prediction** (Primary model)

In [ ]:
from utils.visualization import plot_comparison_grid
from utils.metrics import calculate_all_metrics

batch = next(iter(test_loader))
l_t = batch['L'].to(device)
rgb_t = batch['rgb']

basic_model.eval()
unet_model.eval()

with torch.no_grad():
    pred_basic = lab_to_rgb(l_t, basic_model(l_t))
    pred_unet = lab_to_rgb(l_t, unet_model(l_t))

gt_rgb = (rgb_t.permute(0, 2, 3, 1).cpu().numpy() * 255.0).astype(np.uint8)
l_gray = (l_t[:, 0].cpu().numpy() * 255.0).astype(np.uint8)

# Display first test sample
idx = 0
m_b = calculate_all_metrics(pred_basic[idx], gt_rgb[idx])
m_u = calculate_all_metrics(pred_unet[idx], gt_rgb[idx])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(gt_rgb[idx]); axes[0].set_title("1. Original RGB\n(Ground Truth)"); axes[0].axis('off')
axes[1].imshow(l_gray[idx], cmap='gray'); axes[1].set_title("2. Grayscale Input\n(L Channel)"); axes[1].axis('off')
axes[2].imshow(pred_basic[idx]); axes[2].set_title(f"3. Basic CNN (Baseline)\nPSNR: {m_b['PSNR']:.2f}dB | SSIM: {m_b['SSIM']:.3f}"); axes[2].axis('off')
axes[3].imshow(pred_unet[idx]); axes[3].set_title(f"4. U-Net (Primary)\nPSNR: {m_u['PSNR']:.2f}dB | SSIM: {m_u['SSIM']:.3f}"); axes[3].axis('off')
plt.tight_layout()
plt.show()
print("[STAGE 9 COMPLETE] Visual fidelity verified successfully!")